# Fig.3a variant: glycogene版 薬剤-疾患相関 boxplot

`01_repro_fig3a.ipynb` と同じ枠組みで、**遺伝子空間を glycogene に絞った**版。

- 薬剤側: `RAW.LINCS.GLYCO_GENES_WIDE`（薬剤プロファイル × glycogene 推定発現, 385遺伝子）を化合物ごとに平均。
- 疾患側: CREEDS（do_id平均）を同じ glycogene に制限。
- 共通 glycogene 上で cosine。
- **top/bottom 5%閾値は掛けない**（既に絞った遺伝子パネルなので、glycogene programそのものの信号を見る）。

承認薬マッピングは `RAW.CREEDS.VW_FIG3A_APPROVED_DRUG_MAP` を共用。

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import snowflake.connector as sc

con = sc.connect(
    account='DUETMBM-LL33279', user='KOREEDA', role='ACCOUNTADMIN',
    warehouse='BIOINFORMATICS_XS', authenticator='SNOWFLAKE_JWT',
    private_key_file=os.path.expanduser('~/.ssh/snowflake_rsa_key.pem'),
)
cur = con.cursor()

def q(sql):
    df = cur.execute(sql).fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df

def norm_gene(g):
    return str(g).upper().replace('-', '_')

# GLYCO_GENES_WIDE のメタ列（小文字クォート識別子）
META_LC = {'value','canonical_smiles','cell','cmapid','compound_alias','dose',
           'inchi_key','pertid','pertname','timepoint','sample_id'}

## Step 1. 薬剤 glycogene シグネチャ（化合物ごとに平均、閾値なし）

In [ ]:
df_map = q("SELECT do_id, disease_name, pertname, drug_lc FROM RAW.CREEDS.VW_FIG3A_APPROVED_DRUG_MAP")
drugs_lc = sorted(df_map['drug_lc'].unique())
print(f"diseases={df_map['do_id'].nunique()} drugs={len(drugs_lc)} pairs={len(df_map)}")

inlist = ",".join("'%s'" % d.replace("'", "''") for d in drugs_lc)
# メタ列は小文字クォート、遺伝子列は大文字。SELECT * で取得。
df_drug = q(f'SELECT * FROM RAW.LINCS.GLYCO_GENES_WIDE WHERE LOWER("pertname") IN ({inlist})')
gene_cols = [c for c in df_drug.columns if c not in META_LC]
print(f"profiles={len(df_drug)} glyco_gene_cols={len(gene_cols)}")

# 値はVARCHAR文字列 -> numeric
df_drug[gene_cols] = df_drug[gene_cols].apply(pd.to_numeric, errors='coerce')
df_drug['pert_lc'] = df_drug['pertname'].str.lower()
drug_sig = df_drug.groupby('pert_lc')[gene_cols].mean()
drug_sig.columns = [norm_gene(g) for g in drug_sig.columns]
drug_sig = drug_sig.fillna(0.0)
drug_sig.shape

## Step 2. 疾患 glycogene シグネチャ（CREEDS を do_id 平均）

In [ ]:
df_dis = q("""
  SELECT g.signature_id, g.gene, g.value, m.do_id
  FROM RAW.CREEDS.VW_HUMAN_DISEASE_GENES g
  JOIN RAW.CREEDS.DISEASE_SIGNATURES_META m USING (signature_id)
""")
df_dis['gene'] = df_dis['gene'].map(norm_gene)
n_sig = df_dis.groupby('do_id')['signature_id'].nunique()
dis_sum = df_dis.groupby(['do_id', 'gene'])['value'].sum().reset_index()
dis_sum['avg'] = dis_sum['value'] / dis_sum['do_id'].map(n_sig)
dis_sig = dis_sum.pivot(index='do_id', columns='gene', values='avg').fillna(0.0)
print(f"do_ids={dis_sig.shape[0]} genes={dis_sig.shape[1]}")

## Step 3. 共通 glycogene で cosine 相関

In [ ]:
common = sorted(set(drug_sig.columns) & set(dis_sig.columns))
print(f"common glycogenes = {len(common)}")
D = drug_sig[common]; Y = dis_sig[common]

def cosine(u, v):
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    return np.nan if nu == 0 or nv == 0 else float(np.dot(u, v) / (nu * nv))

rows = []
for _, r in df_map.iterrows():
    do, dz, pl = r['do_id'], r['disease_name'], r['drug_lc']
    if pl in D.index and do in Y.index:
        rows.append((do, dz, pl, cosine(D.loc[pl].values, Y.loc[do].values)))
res = pd.DataFrame(rows, columns=['do_id', 'disease', 'drug', 'cos']).dropna(subset=['cos'])
print(f"pairs={len(res)} diseases={res['do_id'].nunique()}")

out_csv = '../../results/tables/fig3a_glycogene_correlations.csv'
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
res.to_csv(out_csv, index=False)
res.head()

## Step 4. boxplot（中央値昇順、がん=赤/免疫=青）

In [ ]:
CANCER = ['cancer','carcinoma','leukemia','lymphoma','myeloma','melanoma','glioma',
          'sarcoma','astrocytoma','adenoma','tumor','blastoma']
IMMUNE = ['arthritis','psoriasis','asthma','crohn','colitis','lupus','dermatitis',
          'inflammatory','multiple sclerosis']
def cat(name):
    n = name.lower()
    if any(k in n for k in CANCER): return 'cancer'
    if any(k in n for k in IMMUNE): return 'immune'
    return 'other'

med = res.groupby('disease')['cos'].median().sort_values()
order = med.index.tolist()
data = [res.loc[res['disease'] == d, 'cos'].values for d in order]
catcol = {'cancer': '#c0392b', 'immune': '#2471a3', 'other': '#7f8c8d'}
colors = [catcol[cat(d)] for d in order]

fig, ax = plt.subplots(figsize=(15, 5))
bp = ax.boxplot(data, showfliers=True, widths=0.6, patch_artist=True)
for patch, col in zip(bp['boxes'], colors):
    patch.set_facecolor(col); patch.set_alpha(0.55)
ax.axhline(0, color='grey', lw=0.6, ls='--')
ax.set_xticks(range(1, len(order) + 1))
ax.set_xticklabels(order, rotation=90, fontsize=7)
ax.set_ylabel('Cosine correlation (glycogene program)')
ax.set_title(f'Fig3a variant: glycogene-restricted ({len(common)} glycogenes; red=cancer, blue=immune)')

out_fig = '../../results/figures/fig3a_glycogene.png'
os.makedirs(os.path.dirname(out_fig), exist_ok=True)
fig.tight_layout(); fig.savefig(out_fig, dpi=200, bbox_inches='tight')
print('カテゴリ別中央値:'); print(res.assign(cat=res['disease'].map(cat)).groupby('cat')['cos'].median())
plt.show()